# Hubbard chain ground states, checked against exact answers

This notebook solves the Fermi–Hubbard chain with `quantx_engine.dmrg.ground_state` on
the native block-sparse engine, which conserves the up and the down electron numbers
separately, and compares it with exact answers: an exact diagonalisation at 8 sites, and
the Lieb–Wu solution of the infinite chain for the energy and the Mott gap.

It needs the native engine, which the `pip install quantx-engine` wheels carry on Linux,
macOS and Windows, and runs in about a minute.

In [ ]:
import itertools
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import scipy.integrate as si
import scipy.sparse as sp
import scipy.sparse.linalg as sla
import scipy.special as ss

import quantx_engine
from quantx_engine.dmrg import ground_state
from quantx_engine.models import HubbardChain

assert quantx_engine.backend_info()["native"], "this notebook needs the native engine"
U = 4.0      # H = -sum (c+ c + h.c.) + U sum n_up n_dn, open chain, t = 1

## 1. Exact diagonalisation at 8 sites

With 4 up and 4 down electrons on 8 sites the Hamiltonian is a 4900 × 4900 sparse matrix.
Each species is a bit string of occupied sites. On an open chain a nearest-neighbour hop
passes no other electron of its own species, so it carries no fermion sign, and the two
species meet only through the diagonal $U n_\uparrow n_\downarrow$ term.

In [ ]:
def sector_energy(L, U, n_up, n_dn):
    # Lowest energy of the open chain with n_up up and n_dn down electrons.
    ups = [sum(1 << i for i in c) for c in itertools.combinations(range(L), n_up)]
    dns = [sum(1 << i for i in c) for c in itertools.combinations(range(L), n_dn)]

    def hops(states):
        index = {s: k for k, s in enumerate(states)}
        rows, cols = [], []
        for k, s in enumerate(states):
            for i in range(L - 1):
                if ((s >> i) & 1) != ((s >> (i + 1)) & 1):    # one of the two is empty
                    rows.append(index[s ^ (0b11 << i)])
                    cols.append(k)
        return sp.csr_matrix((-np.ones(len(rows)), (rows, cols)), shape=(len(states),) * 2)

    double = np.array([[bin(u & d).count("1") for d in dns] for u in ups], float).ravel()
    H = (sp.kron(hops(ups), sp.identity(len(dns))) + sp.kron(sp.identity(len(ups)), hops(dns))
         + sp.diags(U * double))
    return float(sla.eigsh(H, k=1, which="SA", tol=1e-14)[0][0])


e_ed = sector_energy(8, U, 4, 4)
gs = ground_state(HubbardChain(L=8, U=U), n_up=4, n_dn=4, max_bond=256)
print(f"exact diagonalisation  {e_ed:.12f}")
print(f"DMRG                   {gs.energy:.12f}")
print(f"difference             {abs(gs.energy - e_ed):.1e}")
assert abs(gs.energy - e_ed) < 1e-9

## 2. The infinite chain: Lieb and Wu

Lieb and Wu solved the half-filled chain exactly (1968). With $t = 1$ its ground-state
energy per site and its charge (Mott) gap are one-dimensional integrals:

$$e(U) = -4\int_0^\infty \frac{J_0(\omega)\,J_1(\omega)}{\omega\,\left(1 + e^{\omega U/2}\right)}\,d\omega,
\qquad
\Delta_c(U) = U - 4 + 8\int_0^\infty \frac{J_1(\omega)}{\omega\,\left(1 + e^{\omega U/2}\right)}\,d\omega .$$

They are integrated here one Bessel half-period at a time, out to where the factor
$1/(1 + e^{\omega U/2})$ is below $e^{-50}$.

In [ ]:
def lieb_wu(f, U):
    w_end = 100.0 / U + 50.0
    edges = [k * math.pi for k in range(int(w_end / math.pi) + 2)]
    return sum(si.quad(f, a, b, epsabs=1e-15, epsrel=1e-13)[0] for a, b in zip(edges, edges[1:]))


fermi = lambda w: ss.expit(-w * U / 2)                     # 1 / (1 + e^{wU/2})
e_exact = -4 * lieb_wu(lambda w: ss.j0(w) * ss.j1(w) * fermi(w) / w, U)
gap_exact = U - 4 + 8 * lieb_wu(lambda w: ss.j1(w) * fermi(w) / w, U)
print(f"Lieb-Wu at U = {U:g}:  e = {e_exact:.10f}   charge gap = {gap_exact:.8f}")

## 3. DMRG on longer chains

At half filling the chain is a Mott insulator: its charge excitations have a gap, its
entanglement obeys an area law, and a bond dimension of 128 converges these energies to
about $10^{-7}$ per site. For each length the run takes three sectors: half filling, one
electron more and one fewer, for the gap
$\Delta(L) = E(N+1) + E(N-1) - 2E(N)$.

In [ ]:
sizes = [8, 12, 16, 20, 24]
energy, gap = {}, {}
t0 = time.time()
for L in sizes:
    h = L // 2
    E0, Ep, Em = (ground_state(HubbardChain(L=L, U=U), n_up=h + d, n_dn=h, max_bond=128).energy
                  for d in (0, 1, -1))
    energy[L], gap[L] = E0, Ep + Em - 2 * E0
    print(f"L = {L:2d}:  E/L = {E0 / L:.8f}   gap = {gap[L]:.6f}")
print(f"{time.time() - t0:.0f} s")

### The energy per site

The energy per site of an open chain carries a boundary term of order $1/L$. The
difference $e_\mathrm{bulk}(L) = \left[E(L) - E(L-4)\right]/4$ cancels it, and converges
on the Lieb–Wu value as $1/L^2$.

In [ ]:
bulk = {L: (energy[L] - energy[L - 4]) / 4 for L in sizes[1:]}
for L, e in bulk.items():
    print(f"L = {L:2d}:  e_bulk = {e:.8f}   minus Lieb-Wu = {e - e_exact:+.2e}")

x = np.array([1 / L**2 for L in bulk])
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(x, list(bulk.values()), "o-", label="DMRG, $[E(L) - E(L-4)]/4$")
ax.axhline(e_exact, color="k", ls="--", lw=1, label="Lieb-Wu, exact")
ax.set_xlabel("$1/L^2$"); ax.set_ylabel("energy per site"); ax.set_xlim(left=0)
ax.legend(); plt.tight_layout(); plt.show()
assert abs(bulk[24] - e_exact) < 5e-4

### The Mott gap

The gap converges more slowly, with both $1/L$ and $1/L^2$ terms. A fit of
$\Delta(L) = \Delta_\infty + b/L + c/L^2$ to the four longest chains extrapolates it. The
chains here are kept short so the notebook runs in a minute, and the extrapolation is
correspondingly rough.

In [ ]:
Ls = np.array(sizes[1:], float)
A = np.vstack([np.ones_like(Ls), 1 / Ls, 1 / Ls**2]).T
coef = np.linalg.lstsq(A, np.array([gap[int(L)] for L in Ls]), rcond=None)[0]
print(f"extrapolated gap {coef[0]:.4f}   Lieb-Wu {gap_exact:.4f}   "
      f"relative difference {abs(coef[0] - gap_exact) / gap_exact:.1%}")

xs = np.linspace(0, 1 / 8, 100)
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot([1 / L for L in sizes], [gap[L] for L in sizes], "o", label="DMRG")
ax.plot(xs, coef[0] + coef[1] * xs + coef[2] * xs**2, "-", lw=1, label="fit")
ax.axhline(gap_exact, color="k", ls="--", lw=1, label="Lieb-Wu, exact")
ax.set_xlabel("$1/L$"); ax.set_ylabel("charge gap"); ax.set_xlim(left=0)
ax.legend(); plt.tight_layout(); plt.show()
assert abs(coef[0] - gap_exact) / gap_exact < 0.06

## What this shows

- **Exact at 8 sites:** the DMRG energy is the exact diagonalisation's, to the
  precision printed above.
- **The infinite chain:** the bulk energy approaches the Lieb–Wu value as $1/L^2$, to a
  few $10^{-4}$ at 24 sites, and the gap extrapolates to within a few per cent of the exact
  Mott gap from chains of at most 24 sites.

Longer chains close the gap further: `docs/hubbard_lieb_wu.csv` in the repository goes to
64 sites, where the bulk energy is $8\times10^{-5}$ to $3\times10^{-5}$ from Lieb–Wu for
$U = 2$ to 8.